# 11: Knowledge Graph Explorer

**Purpose**: Understand the structure and contents of the Trajecto trajectory knowledge graph.

---

## What is a Knowledge Graph?

A **knowledge graph** organizes information as a network of connected entities. In Trajecto:

- **Nodes** represent things: Users, Trajectories, POIs (Points of Interest)
- **Edges** represent relationships: "User visited POI", "Trajectory contains POI"

Think of it like a social network, but for travel patterns!

---

## Contents

1. [Graph Overview Dashboard](#1-graph-overview-dashboard)
2. [POI Connections Explorer](#2-poi-connections-explorer)
3. [User-POI Relationships](#3-user-poi-relationships)
4. [Trajectory-POI Patterns](#4-trajectory-poi-patterns)

In [ ]:
# Setup - Run this cell first
import sys

sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import seaborn as sns
from helpers.graph_builder import TrajectoNXGraphBuilder

# Visualization settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
%matplotlib inline

# Initialize graph builder
builder = TrajectoNXGraphBuilder()

---

## 1. Graph Overview Dashboard

Let's start with a bird's eye view of our knowledge graph. How many users, trajectories, and POIs do we have?

In [ ]:
# Build the main graphs we'll use for exploration
print("Building graph views from knowledge graph...")
print("(This may take a moment)\n")

# User-POI bipartite graph
user_poi_graph = builder.build_user_poi_graph()

# Trajectory-POI bipartite graph
traj_poi_graph = builder.build_trajectory_poi_graph()

# User-Trajectory graph
user_traj_graph = builder.build_user_trajectory_graph()

# POI co-visitation graph
poi_covisit_graph = builder.build_poi_covisit_graph()

print("Graph views built successfully!")

In [ ]:
# Count entities by type


def count_by_type(graph, node_type):
    return sum(1 for n, d in graph.nodes(data=True) if d.get('type') == node_type)


# Gather statistics
num_users = count_by_type(user_poi_graph, 'user')
num_pois = count_by_type(user_poi_graph, 'poi')
num_trajectories = count_by_type(traj_poi_graph, 'trajectory')
num_visits = user_poi_graph.number_of_edges()
num_traj_poi_links = traj_poi_graph.number_of_edges()

# Display overview
print("="*60)
print("KNOWLEDGE GRAPH OVERVIEW")
print("="*60)
print("\nEntities:")
print(f"  Users:         {num_users:,}")
print(f"  Trajectories:  {num_trajectories:,}")
print(f"  POIs:          {num_pois:,}")
print("\nRelationships:")
print(f"  User-POI visits:      {num_visits:,}")
print(f"  Trajectory-POI links: {num_traj_poi_links:,}")
print("\nAverages:")
if num_users > 0:
    print(f"  POIs per user:        {num_visits/num_users:.1f}")
if num_trajectories > 0:
    print(f"  POIs per trajectory:  {num_traj_poi_links/num_trajectories:.1f}")

In [ ]:
# Visualize entity distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Entity counts bar chart
entities = ['Users', 'Trajectories', 'POIs']
counts = [num_users, num_trajectories, num_pois]
colors = ['#3498db', '#2ecc71', '#e74c3c']

ax1 = axes[0]
bars = ax1.bar(entities, counts, color=colors, edgecolor='white', linewidth=2)
ax1.set_ylabel('Count', fontsize=12)
ax1.set_title('Entity Distribution', fontsize=14, fontweight='bold')

# Add value labels on bars
for bar, count in zip(bars, counts):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(counts)*0.02,
             f'{count:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Relationship counts
relationships = ['User-POI\nVisits', 'Trajectory-POI\nLinks']
rel_counts = [num_visits, num_traj_poi_links]
rel_colors = ['#9b59b6', '#f39c12']

ax2 = axes[1]
bars2 = ax2.bar(relationships, rel_counts, color=rel_colors, edgecolor='white', linewidth=2)
ax2.set_ylabel('Count', fontsize=12)
ax2.set_title('Relationship Distribution', fontsize=14, fontweight='bold')

for bar, count in zip(bars2, rel_counts):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(rel_counts)*0.02,
             f'{count:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nThe knowledge graph connects users to their visited POIs through trajectories.")

---

## 2. POI Connections Explorer

**Key Question**: Which POIs are frequently visited together?

The **co-visitation graph** connects POIs that appear in the same trajectory. This reveals:
- Natural clusters of related places
- Popular route combinations
- Hidden connections between POIs

In [ ]:
# Analyze POI co-visitation patterns
print("POI Co-Visitation Analysis")
print("="*50)

# Get basic stats
num_poi_nodes = poi_covisit_graph.number_of_nodes()
num_covisit_edges = poi_covisit_graph.number_of_edges()

print(f"\nPOIs with co-visitation data: {num_poi_nodes}")
print(f"Co-visitation connections: {num_covisit_edges}")

if num_poi_nodes > 0:
    density = nx.density(poi_covisit_graph)
    print(f"\nGraph density: {density:.4f}")
    print("  (0 = no connections, 1 = everything connected)")

    if density < 0.1:
        print("  -> Sparse graph: POIs cluster in distinct groups")
    elif density < 0.3:
        print("  -> Moderate connectivity: Some common paths exist")
    else:
        print("  -> Dense graph: Many POI combinations visited")

In [ ]:
# Find most commonly co-visited POI pairs
if poi_covisit_graph.number_of_edges() > 0:
    # Get edges with weights
    edge_weights = [(u, v, d.get('weight', 1))
                    for u, v, d in poi_covisit_graph.edges(data=True)]
    edge_weights.sort(key=lambda x: x[2], reverse=True)

    print("\nTop 10 Most Commonly Co-Visited POI Pairs:")
    print("-"*60)
    print(f"{'POI 1':<25} {'POI 2':<25} {'Times'}")
    print("-"*60)

    for poi1, poi2, weight in edge_weights[:10]:
        # Extract POI names from URIs
        name1 = poi1.split('/')[-1].split('#')[-1][:23]
        name2 = poi2.split('/')[-1].split('#')[-1][:23]
        print(f"{name1:<25} {name2:<25} {weight:>5}")
else:
    print("\nNo co-visitation data available.")

In [ ]:
# Visualize co-visitation network (top connections only)
if poi_covisit_graph.number_of_nodes() > 0 and poi_covisit_graph.number_of_edges() > 0:
    # Create subgraph with top weighted edges for visualization
    edges_by_weight = sorted(poi_covisit_graph.edges(data=True),
                              key=lambda x: x[2].get('weight', 0),
                              reverse=True)

    # Take top edges for clarity
    top_n = min(50, len(edges_by_weight))
    top_edges = edges_by_weight[:top_n]

    # Build subgraph
    G_viz = nx.Graph()
    for u, v, d in top_edges:
        G_viz.add_edge(u, v, **d)

    if G_viz.number_of_nodes() > 2:
        plt.figure(figsize=(12, 10))

        # Layout
        pos = nx.spring_layout(G_viz, k=2, iterations=50, seed=42)

        # Node sizes by degree
        degrees = dict(G_viz.degree())
        node_sizes = [100 + degrees[n] * 50 for n in G_viz.nodes()]

        # Edge weights for width
        weights = [G_viz[u][v].get('weight', 1) for u, v in G_viz.edges()]
        max_w = max(weights) if weights else 1
        edge_widths = [1 + (w / max_w) * 4 for w in weights]

        # Draw
        nx.draw_networkx_edges(G_viz, pos, width=edge_widths, alpha=0.5, edge_color='gray')
        nx.draw_networkx_nodes(G_viz, pos, node_size=node_sizes,
                               node_color='#3498db', alpha=0.8)

        # Labels for high-degree nodes
        labels = {n: n.split('/')[-1].split('#')[-1][:15]
                  for n in G_viz.nodes() if degrees[n] >= 3}
        nx.draw_networkx_labels(G_viz, pos, labels, font_size=8)

        plt.title('POI Co-Visitation Network\n(Top connections, node size = popularity)',
                  fontsize=14, fontweight='bold')
        plt.axis('off')
        plt.tight_layout()
        plt.show()

        print(f"\nShowing {G_viz.number_of_nodes()} POIs with {G_viz.number_of_edges()} co-visitation links.")
        print("Larger nodes = more connections (popular on routes)")
    else:
        print("Not enough data for visualization.")
else:
    print("No co-visitation data available for visualization.")

---

## 3. User-POI Relationships

**Key Question**: Who visits what?

The user-POI graph is **bipartite** (two types of nodes that only connect across types):
- Users on one side
- POIs on the other side
- Edges represent visits

In [ ]:
# Analyze user activity distribution
if user_poi_graph.number_of_nodes() > 0:
    # Get user degrees (number of POIs visited)
    user_nodes = [n for n, d in user_poi_graph.nodes(data=True) if d.get('type') == 'user']
    user_degrees = [user_poi_graph.degree(u) for u in user_nodes]

    # Get POI degrees (number of visitors)
    poi_nodes = [n for n, d in user_poi_graph.nodes(data=True) if d.get('type') == 'poi']
    poi_degrees = [user_poi_graph.degree(p) for p in poi_nodes]

    print("User Activity Statistics:")
    print("="*50)
    if user_degrees:
        print(f"  Average POIs per user: {np.mean(user_degrees):.1f}")
        print(f"  Most active user: {max(user_degrees)} POIs")
        print(f"  Least active user: {min(user_degrees)} POIs")

    print("\nPOI Popularity Statistics:")
    print("="*50)
    if poi_degrees:
        print(f"  Average visitors per POI: {np.mean(poi_degrees):.1f}")
        print(f"  Most popular POI: {max(poi_degrees)} visitors")
        print(f"  Least visited POI: {min(poi_degrees)} visitors")

In [ ]:
# Visualize distributions
if user_poi_graph.number_of_nodes() > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # User activity distribution
    if user_degrees:
        ax1 = axes[0]
        ax1.hist(user_degrees, bins=30, color='#3498db', edgecolor='white', alpha=0.8)
        ax1.axvline(np.mean(user_degrees), color='red', linestyle='--',
                    label=f'Mean: {np.mean(user_degrees):.1f}')
        ax1.set_xlabel('Number of POIs Visited', fontsize=12)
        ax1.set_ylabel('Number of Users', fontsize=12)
        ax1.set_title('User Activity Distribution\n(How many POIs does each user visit?)',
                      fontsize=14, fontweight='bold')
        ax1.legend()

    # POI popularity distribution
    if poi_degrees:
        ax2 = axes[1]
        ax2.hist(poi_degrees, bins=30, color='#e74c3c', edgecolor='white', alpha=0.8)
        ax2.axvline(np.mean(poi_degrees), color='blue', linestyle='--',
                    label=f'Mean: {np.mean(poi_degrees):.1f}')
        ax2.set_xlabel('Number of Visitors', fontsize=12)
        ax2.set_ylabel('Number of POIs', fontsize=12)
        ax2.set_title('POI Popularity Distribution\n(How many visitors does each POI get?)',
                      fontsize=14, fontweight='bold')
        ax2.legend()

    plt.tight_layout()
    plt.show()

    print("\nInsight: Most POIs follow a 'long tail' distribution - a few very popular places,")
    print("and many niche locations with few visitors.")

In [ ]:
# Find most popular POIs
if poi_nodes and poi_degrees:
    poi_popularity = list(zip(poi_nodes, poi_degrees))
    poi_popularity.sort(key=lambda x: x[1], reverse=True)

    print("Top 15 Most Popular POIs:")
    print("="*50)
    print(f"{'Rank':<6} {'POI Name':<35} {'Visitors'}")
    print("-"*50)

    for i, (poi, visitors) in enumerate(poi_popularity[:15], 1):
        name = poi.split('/')[-1].split('#')[-1][:33]
        print(f"{i:<6} {name:<35} {visitors:>8}")

---

## 4. Trajectory-POI Patterns

**Key Question**: How do trajectories connect to POIs?

A **trajectory** is a sequence of POI visits by a user. Understanding trajectory-POI connections reveals:
- Typical trip lengths
- POI combinations in routes
- Travel patterns

In [ ]:
# Analyze trajectory sizes
if traj_poi_graph.number_of_nodes() > 0:
    traj_nodes = [n for n, d in traj_poi_graph.nodes(data=True) if d.get('type') == 'trajectory']
    traj_sizes = [traj_poi_graph.degree(t) for t in traj_nodes]

    print("Trajectory Size Analysis:")
    print("="*50)
    if traj_sizes:
        print(f"  Total trajectories: {len(traj_sizes)}")
        print(f"  Average POIs per trajectory: {np.mean(traj_sizes):.1f}")
        print(f"  Median POIs per trajectory: {np.median(traj_sizes):.1f}")
        print(f"  Largest trajectory: {max(traj_sizes)} POIs")
        print(f"  Smallest trajectory: {min(traj_sizes)} POIs")

        # Categorize trajectories
        short = sum(1 for s in traj_sizes if s <= 3)
        medium = sum(1 for s in traj_sizes if 4 <= s <= 7)
        long_trips = sum(1 for s in traj_sizes if s > 7)

        print("\nTrajectory Categories:")
        print(f"  Short trips (1-3 POIs): {short} ({short/len(traj_sizes)*100:.1f}%)")
        print(f"  Medium trips (4-7 POIs): {medium} ({medium/len(traj_sizes)*100:.1f}%)")
        print(f"  Long trips (8+ POIs): {long_trips} ({long_trips/len(traj_sizes)*100:.1f}%)")

In [ ]:
# Visualize trajectory size distribution
if traj_poi_graph.number_of_nodes() > 0 and traj_sizes:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Histogram
    ax1 = axes[0]
    ax1.hist(traj_sizes, bins=range(1, max(traj_sizes)+2),
             color='#2ecc71', edgecolor='white', alpha=0.8, align='left')
    ax1.axvline(np.mean(traj_sizes), color='red', linestyle='--',
                label=f'Mean: {np.mean(traj_sizes):.1f}')
    ax1.set_xlabel('Number of POIs in Trajectory', fontsize=12)
    ax1.set_ylabel('Number of Trajectories', fontsize=12)
    ax1.set_title('Trajectory Size Distribution', fontsize=14, fontweight='bold')
    ax1.legend()

    # Pie chart for categories
    ax2 = axes[1]
    categories = ['Short (1-3)', 'Medium (4-7)', 'Long (8+)']
    sizes = [short, medium, long_trips]
    colors_pie = ['#3498db', '#f39c12', '#e74c3c']
    explode = (0.05, 0.05, 0.05)

    ax2.pie(sizes, explode=explode, labels=categories, colors=colors_pie,
            autopct='%1.1f%%', startangle=90, shadow=False)
    ax2.set_title('Trajectory Categories', fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.show()

In [ ]:
# Get POI sequences for sample trajectories
print("Sample Trajectory POI Sequences:")
print("="*60)

sequences = builder.get_trajectory_poi_sequences()

if sequences:
    # Show a few example trajectories
    sample_trajs = list(sequences.items())[:5]

    for traj_uri, pois in sample_trajs:
        traj_name = traj_uri.split('/')[-1].split('#')[-1][:30]
        print(f"\n{traj_name}:")
        print(f"  POIs visited: {len(pois)}")
        print("  Sequence:")
        for i, poi in enumerate(pois[:5], 1):
            poi_name = poi.split('/')[-1].split('#')[-1][:40]
            print(f"    {i}. {poi_name}")
        if len(pois) > 5:
            print(f"    ... and {len(pois)-5} more")
else:
    print("No trajectory sequences available.")

---

## Summary

In this notebook, we explored the Trajecto knowledge graph structure:

### Key Findings

1. **Graph Structure**: The knowledge graph connects Users -> Trajectories -> POIs

2. **POI Co-visitation**: Some POIs are frequently visited together, revealing natural routes and clusters

3. **User-POI Patterns**: POI popularity follows a long-tail distribution (few popular, many niche)

4. **Trajectory Patterns**: Most trajectories contain a moderate number of POIs

### Next Steps

- Notebook 14: graph metrics (centrality, communities, paths)
- Notebook 21: KG-native analytics written back to the KG
- Notebook 15: per-trajectory profile cards

In [ ]:
print("\n" + "="*60)
print("KNOWLEDGE GRAPH EXPLORATION COMPLETE")
print("="*60)
print("\nGraph Summary:")
print(f"  {num_users} users")
print(f"  {num_trajectories} trajectories")
print(f"  {num_pois} points of interest")
print("\nNext: 14-graph-analytics.ipynb for deeper analysis")